amazon Deletion

In [ ]:
import pandas as pd
import os
import random
import torch
import numpy as np

# Set random seed
random.seed(2024)

# ==================== Configuration Parameters ====================
input_path = 'C:/Users/THINK BOOK-16/Desktop/beauty/'  # Directory containing inter.csv for toy dataset
output_path = 'C:/Users/THINK BOOK-16/Desktop/beauty-processed-delete-10/'
dataset_name = 'beauty'

user_threshold = 5
item_threshold = 5
max_seq_len = 50

# ==================== Delete 10% version ====================
DELETE_RATIO = 0.10  # Delete 10% of interactions

# ==================== Helper Functions ====================
def delete_interactions_guarantee_thresholds(dataset, delete_ratio=0.05):
    """
    Delete interactions while ensuring:
    1. All users have interaction count ≥ user_threshold
    2. All items have interaction count ≥ item_threshold
    3. Delete entire interaction records
    4. Precisely control total deletion ratio to delete_ratio
    """
    print(f"\n  🔧 Starting to delete interactions (ensuring user and item thresholds)...")
    
    # Get original data information
    original_size = len(dataset)
    target_deletions = int(original_size * delete_ratio)
    
    print(f"  Original interactions: {original_size}")
    print(f"  Target deletions: {target_deletions} (expected deletion ratio: {delete_ratio*100:.1f}%)")
    
    # Get initial user and item counts
    user_counts = dataset['user_id'].value_counts().to_dict()
    item_counts = dataset['item_id'].value_counts().to_dict()
    
    print(f"  Initial users: {len(user_counts)}")
    print(f"  Initial items: {len(item_counts)}")
    print(f"  Initial user min interactions: {min(user_counts.values())}")
    print(f"  Initial item min interactions: {min(item_counts.values())}")
    
    # Copy dataset
    deleted_dataset = dataset.copy()
    
    # Track current interaction counts for each user and item (dynamically updated)
    current_user_counts = user_counts.copy()
    current_item_counts = item_counts.copy()
    
    # Phase 1: Collect all possible deletable interactions
    print("  Phase 1: Collecting candidate deletion positions...")
    candidate_indices = []
    
    for idx in deleted_dataset.index:
        user = deleted_dataset.at[idx, 'user_id']
        item = deleted_dataset.at[idx, 'item_id']
        
        # Check if thresholds would still be satisfied after deletion
        if (current_user_counts[user] - 1 >= user_threshold and 
            current_item_counts[item] - 1 >= item_threshold):
            candidate_indices.append(idx)
    
    print(f"  Found {len(candidate_indices)} candidate deletion positions")
    
    # Randomly shuffle candidate positions
    random.shuffle(candidate_indices)
    
    # Phase 2: Perform deletions until target is reached
    print("  Phase 2: Executing deletions...")
    total_deletions = 0
    indices_to_delete = []
    
    for idx in candidate_indices:
        if total_deletions >= target_deletions:
            break
            
        user = deleted_dataset.at[idx, 'user_id']
        item = deleted_dataset.at[idx, 'item_id']
        
        # Double-check deletion safety (since counts have been updated)
        if (current_user_counts[user] - 1 >= user_threshold and 
            current_item_counts[item] - 1 >= item_threshold):
            
            # Record index to delete
            indices_to_delete.append(idx)
            
            # Update counts
            current_user_counts[user] -= 1
            current_item_counts[item] -= 1
            
            total_deletions += 1
            
            # Show progress
            if total_deletions % 500 == 0 or total_deletions == target_deletions:
                progress = total_deletions / target_deletions * 100
                print(f"    Progress: {total_deletions}/{target_deletions} ({progress:.1f}%)")
    
    # Perform batch deletion
    if indices_to_delete:
        deleted_dataset = deleted_dataset.drop(indices_to_delete)
        deleted_dataset = deleted_dataset.reset_index(drop=True)
    
    # Calculate actual deletion ratio
    actual_deletions = total_deletions
    actual_delete_ratio = actual_deletions / original_size
    
    print(f"\n  📊 Deletion completed:")
    print(f"    Target deletions: {target_deletions}")
    print(f"    Actual deletions: {actual_deletions}")
    print(f"    Actual deletion ratio: {actual_delete_ratio*100:.2f}%")
    
    # Verify final user and item counts
    final_user_counts = deleted_dataset['user_id'].value_counts()
    final_item_counts = deleted_dataset['item_id'].value_counts()
    
    print(f"\n  Final verification:")
    print(f"    Final users: {len(final_user_counts)}")
    print(f"    Final items: {len(final_item_counts)}")
    print(f"    Final user min interactions: {final_user_counts.min()}")
    print(f"    Final item min interactions: {final_item_counts.min()}")
    
    # Check if all users and items satisfy thresholds
    users_below_threshold = final_user_counts[final_user_counts < user_threshold]
    items_below_threshold = final_item_counts[final_item_counts < item_threshold]
    
    if len(users_below_threshold) == 0:
        print(f"  ✅ All users have interaction count ≥ {user_threshold}")
    else:
        print(f"  ❌ {len(users_below_threshold)} users have interaction count < {user_threshold}")
        return None, 0, 0
    
    if len(items_below_threshold) == 0:
        print(f"  ✅ All items have interaction count ≥ {item_threshold}")
    else:
        print(f"  ❌ {len(items_below_threshold)} items have interaction count < {item_threshold}")
        return None, 0, 0
    
    return deleted_dataset, total_deletions, original_size

def truncate_or_pad(seq):
    """Truncate or pad sequence to fixed length (maintain original logic)"""
    cur_seq_len = len(seq)
    if cur_seq_len > max_seq_len:
        return seq[-max_seq_len:], max_seq_len
    else:
        PAD = 0
        return seq + [PAD] * (max_seq_len - cur_seq_len), cur_seq_len

def ensure_python_types(data):
    """Ensure all data is Python native types instead of numpy types"""
    if isinstance(data, np.integer):
        return int(data)
    elif isinstance(data, np.floating):
        return float(data)
    elif isinstance(data, np.ndarray):
        return data.tolist()
    elif isinstance(data, list):
        return [ensure_python_types(item) for item in data]
    elif isinstance(data, dict):
        return {key: ensure_python_types(value) for key, value in data.items()}
    else:
        return data

# ==================== Main Process ====================

print("="*60)
print("Starting Toy Dataset Processing (10% deletion, ensuring user and item thresholds)")
print("="*60)

# 1. Load inter.csv file
print("\n1. Loading inter.csv file...")
inter_file = os.path.join(input_path, 'inter.csv')

if not os.path.exists(inter_file):
    print(f"Error: Cannot find file {inter_file}")
    exit(1)

# Read inter.csv file
dataset = pd.read_csv(inter_file)

print(f"Original data statistics:")
print(f"  Data shape: {dataset.shape}")
print(f"  Column names: {dataset.columns.tolist()}")
print(f"  Number of users: {dataset['user_id'].nunique()}")
print(f"  Number of items: {dataset['item_id'].nunique()}")
print(f"  Number of interactions: {len(dataset)}")

# 2. Filter dataset (based on interaction frequency)
print("\n2. Filtering dataset...")
filtered_dataset = dataset.copy()
while True:
    ori_len = len(filtered_dataset)
    
    # Filter users with less than user_threshold interactions
    filtered_dataset = filtered_dataset[filtered_dataset['user_id'].map(filtered_dataset['user_id'].value_counts()) >= user_threshold]
    # Filter items with less than item_threshold interactions
    filtered_dataset = filtered_dataset[filtered_dataset['item_id'].map(filtered_dataset['item_id'].value_counts()) >= item_threshold]
    
    if len(filtered_dataset) == ori_len:
        break

print(f"\nFiltered data statistics:")
print(f"  Number of users: {filtered_dataset['user_id'].nunique()}")
print(f"  Number of items: {filtered_dataset['item_id'].nunique()}")
print(f"  Number of interactions: {len(filtered_dataset)}")
print(f"  User interaction range: {filtered_dataset['user_id'].value_counts().min()} ~ {filtered_dataset['user_id'].value_counts().max()}")
print(f"  Item interaction range: {filtered_dataset['item_id'].value_counts().min()} ~ {filtered_dataset['item_id'].value_counts().max()}")

# 3. Remap IDs (consistent with original code)
print("\n3. Remapping IDs...")
all_user = filtered_dataset.user_id
all_item = filtered_dataset.item_id

user_id, user_token = pd.factorize(all_user)
item_id, item_token = pd.factorize(all_item)

num_users = len(user_token) + 1  # 0 id is for PAD
num_items = len(item_token) + 1  # 0 id is for PAD

user_mapping_dict = {_: idx + 1 for idx, _ in enumerate(user_token)}  # 0 id is for PAD
item_mapping_dict = {_: idx + 1 for idx, _ in enumerate(item_token)}  # 0 id is for PAD

print(f"User mapping: {user_token.shape}")
print(f"Item mapping: {item_token.shape}")

filtered_dataset['user_id'] = filtered_dataset['user_id'].apply(lambda x: user_mapping_dict[x])
filtered_dataset['item_id'] = filtered_dataset['item_id'].apply(lambda x: item_mapping_dict[x])

# Check if domain column exists, if not add one
if 'domain' not in filtered_dataset.columns:
    print("Note: No domain column in data, adding default domain=0")
    filtered_dataset['domain'] = 0

# Ensure domain is integer type
filtered_dataset['domain'] = filtered_dataset['domain'].astype(int)

# Save clean item list and counts
clean_users = sorted(filtered_dataset['user_id'].unique())
clean_items = sorted(filtered_dataset['item_id'].unique())
clean_user_counts = filtered_dataset['user_id'].value_counts().to_dict()
clean_item_counts = filtered_dataset['item_id'].value_counts().to_dict()
print(f"Clean user count: {len(clean_users)}")
print(f"Clean item count: {len(clean_items)}")
print(f"Clean user ID range: {clean_users[0]} ~ {clean_users[-1]}")
print(f"Clean item ID range: {clean_items[0]} ~ {clean_items[-1]}")
print(f"Clean minimum user interactions: {min(clean_user_counts.values())}")
print(f"Clean minimum item interactions: {min(clean_item_counts.values())}")

# 4. Create output directory
os.makedirs(output_path, exist_ok=True)

# 5. Process clean data
print(f"\n{'='*40}")
print(f"Processing clean data...")
print(f"{'='*40}")

clean_dataset = filtered_dataset.copy()
clean_type = 'clean'

# Create output directory
clean_dir = os.path.join(output_path, dataset_name, clean_type)
os.makedirs(clean_dir, exist_ok=True)

# Save inter.csv
csv_path = os.path.join(clean_dir, 'inter.csv')
clean_dataset.to_csv(csv_path, sep=',', index=None)
print(f"  Saving {clean_type} interaction data to: {csv_path}")

# Generate sequence data following original logic
print(f"  Generating sequence data...")

# Sort by user and time (original code logic)
clean_dataset_sorted = clean_dataset.sort_values(by=['user_id', 'timestamp'])

# ==================== Generate seq2pat_data.pth ====================
def to_list(x):
    return list(x)[:-2]  # Remove last 2 interactions, maintain original logic

user_group_for_seq2pat = clean_dataset_sorted.groupby('user_id')['item_id'].apply(to_list)

# Filter out empty lists
seq2pat_data = [seq for seq in user_group_for_seq2pat.tolist() if len(seq) > 0]

# Ensure data types are Python native types
seq2pat_data = ensure_python_types(seq2pat_data)

# Save seq2pat_data.pth
seq2pat_path = os.path.join(clean_dir, 'seq2pat_data.pth')
torch.save(seq2pat_data, seq2pat_path)
print(f"  Generated {clean_type}/seq2pat_data.pth, valid sequences: {len(seq2pat_data)}")
# ==================== seq2pat_data generation ends ====================

# Continue generating train/val/test sequences
user_group = clean_dataset_sorted.groupby('user_id')['item_id'].apply(list)

# Initialize lists
train, val, test = [], [], []

PAD = 0

# Process each user
for user_id, user_seq in list(zip(user_group.index, user_group.tolist())):
    # Ensure user_id is Python int type
    user_id = int(user_id)
    
    # Truncate to maximum sequence length
    user_seq = user_seq[-max_seq_len:]
    
    # Skip users with sequences too short
    if len(user_seq) < 3:
        continue
        
    # Ensure item_id in user_seq is Python int type
    user_seq = [int(item) for item in user_seq]
        
    # ------ Test sample ------------
    history, seq_len = truncate_or_pad(user_seq[:-1])
    target_data = user_seq[-1]
    label = 1
    
    # Domain handling: following original code logic, domain_id is [0] * max_seq_len
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    test.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Validation sample -------------
    history, seq_len = truncate_or_pad(user_seq[:-2])
    target_data = user_seq[-2]
    label = 1
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    val.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Training sample -----------
    # Maintain original logic
    history, seq_len = truncate_or_pad(user_seq[:-3])
    target_data, _ = truncate_or_pad(user_seq[-seq_len-2:-2])
    label = [1] * seq_len + [PAD] * (max_seq_len - seq_len)
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    train.append([user_id, history, target_data, seq_len, label, domain_id])

# Ensure all data are Python native types
train = ensure_python_types(train)
val = ensure_python_types(val)
test = ensure_python_types(test)

# Save sequence data
torch.save(train, os.path.join(clean_dir, 'train.pth'))
torch.save(val, os.path.join(clean_dir, 'val.pth'))
torch.save(test, os.path.join(clean_dir, 'test.pth'))

print(f"  Saved sequence data: training={len(train)}, validation={len(val)}, test={len(test)}")

# 6. Generate 10% deletion data
print(f"\n{'='*40}")
print(f"Processing 10% deletion data (ensuring user and item thresholds)...")
print(f"{'='*40}")

# Generate deletion data
result = delete_interactions_guarantee_thresholds(
    filtered_dataset.copy(), 
    delete_ratio=DELETE_RATIO
)

if result[0] is None:
    print("❌ Deletion failed, cannot satisfy all conditions!")
    # Create an empty deletion dataset but mark as failed
    deleted_type = f'delete_{int(DELETE_RATIO*100)}_FAILED'
    deleted_dir = os.path.join(output_path, dataset_name, deleted_type)
    os.makedirs(deleted_dir, exist_ok=True)
    
    # Save error message
    with open(os.path.join(deleted_dir, 'ERROR.txt'), 'w') as f:
        f.write(f"Deletion operation failed: Cannot delete {DELETE_RATIO*100}% of interactions while ensuring all users and items have interaction count ≥ {item_threshold}.")
    
    print(f"  Created error marker directory: {deleted_dir}")
else:
    # Unpack result
    deleted_dataset, total_deletions, original_size = result
    
    deleted_type = f'delete_{int(DELETE_RATIO*100)}'
    
    # Create output directory
    deleted_dir = os.path.join(output_path, dataset_name, deleted_type)
    os.makedirs(deleted_dir, exist_ok=True)
    
    # Save inter.csv
    deleted_csv_path = os.path.join(deleted_dir, 'inter.csv')
    deleted_dataset.to_csv(deleted_csv_path, sep=',', index=None)
    print(f"  Saving {deleted_type} interaction data to: {deleted_csv_path}")
    
    # Generate sequence data following original logic
    print(f"  Generating sequence data...")
    
    # Sort by user and time (original code logic)
    deleted_dataset_sorted = deleted_dataset.sort_values(by=['user_id', 'timestamp'])
    
    # ==================== Generate seq2pat_data.pth ====================
    user_group_for_seq2pat_deleted = deleted_dataset_sorted.groupby('user_id')['item_id'].apply(to_list)
    
    # Filter out empty lists
    seq2pat_data_deleted = [seq for seq in user_group_for_seq2pat_deleted.tolist() if len(seq) > 0]
    
    # Ensure data types are Python native types
    seq2pat_data_deleted = ensure_python_types(seq2pat_data_deleted)
    
    # Save seq2pat_data.pth
    seq2pat_path_deleted = os.path.join(deleted_dir, 'seq2pat_data.pth')
    torch.save(seq2pat_data_deleted, seq2pat_path_deleted)
    print(f"  Generated {deleted_type}/seq2pat_data.pth, valid sequences: {len(seq2pat_data_deleted)}")
    # ==================== seq2pat_data generation ends ====================
    
    # Continue generating train/val/test sequences
    user_group_deleted = deleted_dataset_sorted.groupby('user_id')['item_id'].apply(list)
    
    # Initialize lists
    train_deleted, val_deleted, test_deleted = [], [], []
    
    # Process each user
    for user_id, user_seq in list(zip(user_group_deleted.index, user_group_deleted.tolist())):
        # Ensure user_id is Python int type
        user_id = int(user_id)
        
        # Truncate to maximum sequence length
        user_seq = user_seq[-max_seq_len:]
        
        # Skip users with sequences too short
        if len(user_seq) < 3:
            continue
            
        # Ensure user_seq中的item_id是Python int类型
        user_seq = [int(item) for item in user_seq]
            
        # ------ Test sample ------------
        history, seq_len = truncate_or_pad(user_seq[:-1])
        target_data = user_seq[-1]
        label = 1
        
        # Domain handling: following original code logic, domain_id is [0] * max_seq_len
        domain_id = [0] * max_seq_len
        
        # Ensure data types
        history = ensure_python_types(history)
        target_data = ensure_python_types(target_data)
        seq_len = ensure_python_types(seq_len)
        label = ensure_python_types(label)
        domain_id = ensure_python_types(domain_id)
        
        test_deleted.append([user_id, history, target_data, seq_len, label, domain_id, history])
        
        # ------ Validation sample -------------
        history, seq_len = truncate_or_pad(user_seq[:-2])
        target_data = user_seq[-2]
        label = 1
        domain_id = [0] * max_seq_len
        
        # Ensure data types
        history = ensure_python_types(history)
        target_data = ensure_python_types(target_data)
        seq_len = ensure_python_types(seq_len)
        label = ensure_python_types(label)
        domain_id = ensure_python_types(domain_id)
        
        val_deleted.append([user_id, history, target_data, seq_len, label, domain_id, history])
        
        # ------ Training sample -----------
        history, seq_len = truncate_or_pad(user_seq[:-3])
        target_data, _ = truncate_or_pad(user_seq[-seq_len-2:-2])
        label = [1] * seq_len + [PAD] * (max_seq_len - seq_len)
        domain_id = [0] * max_seq_len
        
        # Ensure data types
        history = ensure_python_types(history)
        target_data = ensure_python_types(target_data)
        seq_len = ensure_python_types(seq_len)
        label = ensure_python_types(label)
        domain_id = ensure_python_types(domain_id)
        
        train_deleted.append([user_id, history, target_data, seq_len, label, domain_id])
    
    # Ensure all data are Python native types
    train_deleted = ensure_python_types(train_deleted)
    val_deleted = ensure_python_types(val_deleted)
    test_deleted = ensure_python_types(test_deleted)
    
    # Save sequence data
    torch.save(train_deleted, os.path.join(deleted_dir, 'train.pth'))
    torch.save(val_deleted, os.path.join(deleted_dir, 'val.pth'))
    torch.save(test_deleted, os.path.join(deleted_dir, 'test.pth'))
    
    print(f"  Saved sequence data: training={len(train_deleted)}, validation={len(val_deleted)}, test={len(test_deleted)}")

print("\n" + "="*60)
print("Data processing completed!")
print("="*60)

# 7. Final verification
print(f"\n{'='*40}")
print("Final verification results:")
print(f"{'='*40}")

if result[0] is not None:
    # Verify data integrity
    df_clean = pd.read_csv(csv_path)
    df_deleted = pd.read_csv(deleted_csv_path)
    
    print(f"Clean data: {len(df_clean)} rows, {df_clean['user_id'].nunique()} users, {df_clean['item_id'].nunique()} items")
    print(f"Delete data: {len(df_deleted)} rows, {df_deleted['user_id'].nunique()} users, {df_deleted['item_id'].nunique()} items")
    
    # Verify user thresholds
    clean_user_min = df_clean['user_id'].value_counts().min()
    clean_item_min = df_clean['item_id'].value_counts().min()
    deleted_user_min = df_deleted['user_id'].value_counts().min()
    deleted_item_min = df_deleted['item_id'].value_counts().min()
    
    print(f"\n  User min interactions: Clean={clean_user_min}, Delete={deleted_user_min}")
    print(f"  Item min interactions: Clean={clean_item_min}, Delete={deleted_item_min}")
    
    if deleted_user_min >= user_threshold:
        print(f"  All users have interaction count ≥ {user_threshold}")
    else:
        print(f"  Some users have interaction count < {user_threshold}")
    
    if deleted_item_min >= item_threshold:
        print(f"  All items have interaction count ≥ {item_threshold}")
    else:
        print(f"  Some items have interaction count < {item_threshold}")
    
    # Calculate actual deletion ratio
    actual_delete_ratio = (len(df_clean) - len(df_deleted)) / len(df_clean)
    print(f"\nDeletion ratio statistics:")
    print(f"  Target deletion ratio: {DELETE_RATIO*100:.1f}%")
    print(f"  Actual deletion ratio: {actual_delete_ratio*100:.2f}%")
    print(f"  Absolute error: {abs(actual_delete_ratio - DELETE_RATIO)*100:.3f}%")

print(f"\nOutput directory structure:")
print(f"{output_path}/")
print(f"└── {dataset_name}/")
print(f"    ├── clean/")
print(f"    │   ├── inter.csv")
print(f"    │   ├── seq2pat_data.pth")
print(f"    │   ├── train.pth")
print(f"    │   ├── val.pth")
print(f"    │   └── test.pth")

if result[0] is not None:
    print(f"    └── delete_10/")
    print(f"        ├── inter.csv")
    print(f"        ├── seq2pat_data.pth")
    print(f"        ├── train.pth")
    print(f"        ├── val.pth")
    print(f"        └── test.pth")

print(f"\nKey guarantees:")
print(f"1. All users have interaction count ≥ {user_threshold}")
print(f"2. All items have interaction count ≥ {item_threshold}")
print(f"3. Precisely controlled deletion ratio ≈ 10%")
print(f"4. Delete entire interaction records")
print(f"5. Domain handled according to original code logic (fixed to 0)")
print(f"6. All data are Python native types, avoiding numpy type issues")

print("\nToy dataset deletion experiment ready!")

yelp Deletion

In [ ]:
import pandas as pd
import os
import random
import torch
import numpy as np

# Set random seed
random.seed(2024)

# ==================== Configuration Parameters ====================
input_path = 'C:/Users/THINK BOOK-16/Desktop/yelp/'  # Directory containing yelp's inter.csv
output_path = 'C:/Users/THINK BOOK-16/Desktop/yelp-processed-delete-10/'
dataset_name = 'yelp'

user_threshold = 5
item_threshold = 5
max_seq_len = 50

# ==================== Delete 10% version ====================
DELETE_RATIO = 0.10  # Delete 10% of interactions

# ==================== Helper Functions ====================
def delete_interactions_guarantee_thresholds(dataset, delete_ratio=0.05):
    """
    Delete interactions while ensuring:
    1. All users have interaction count ≥ user_threshold
    2. All items have interaction count ≥ item_threshold
    3. Delete entire interaction records
    4. Precisely control total deletion ratio to delete_ratio
    """
    print(f"\n  🔧 Starting to delete interactions (ensuring user and item thresholds)...")
    
    # Get original data information
    original_size = len(dataset)
    target_deletions = int(original_size * delete_ratio)
    
    print(f"  Original interactions: {original_size}")
    print(f"  Target deletions: {target_deletions} (expected deletion ratio: {delete_ratio*100:.1f}%)")
    
    # Get initial user and item counts
    user_counts = dataset['user_id'].value_counts().to_dict()
    item_counts = dataset['item_id'].value_counts().to_dict()
    
    print(f"  Initial users: {len(user_counts)}")
    print(f"  Initial items: {len(item_counts)}")
    print(f"  Initial user min interactions: {min(user_counts.values())}")
    print(f"  Initial item min interactions: {min(item_counts.values())}")
    
    # Copy dataset
    deleted_dataset = dataset.copy()
    
    # Track current interaction counts for each user and item (dynamically updated)
    current_user_counts = user_counts.copy()
    current_item_counts = item_counts.copy()
    
    # Phase 1: Collect all possible deletable interactions
    print("  Phase 1: Collecting candidate deletion positions...")
    candidate_indices = []
    
    for idx in deleted_dataset.index:
        user = deleted_dataset.at[idx, 'user_id']
        item = deleted_dataset.at[idx, 'item_id']
        
        # Check if thresholds would still be satisfied after deletion
        if (current_user_counts[user] - 1 >= user_threshold and 
            current_item_counts[item] - 1 >= item_threshold):
            candidate_indices.append(idx)
    
    print(f"  Found {len(candidate_indices)} candidate deletion positions")
    
    # Randomly shuffle candidate positions
    random.shuffle(candidate_indices)
    
    # Phase 2: Perform deletions until target is reached
    print("  Phase 2: Executing deletions...")
    total_deletions = 0
    indices_to_delete = []
    
    for idx in candidate_indices:
        if total_deletions >= target_deletions:
            break
            
        user = deleted_dataset.at[idx, 'user_id']
        item = deleted_dataset.at[idx, 'item_id']
        
        # Double-check deletion safety (since counts have been updated)
        if (current_user_counts[user] - 1 >= user_threshold and 
            current_item_counts[item] - 1 >= item_threshold):
            
            # Record index to delete
            indices_to_delete.append(idx)
            
            # Update counts
            current_user_counts[user] -= 1
            current_item_counts[item] -= 1
            
            total_deletions += 1
            
            # Show progress
            if total_deletions % 500 == 0 or total_deletions == target_deletions:
                progress = total_deletions / target_deletions * 100
                print(f"    Progress: {total_deletions}/{target_deletions} ({progress:.1f}%)")
    
    # Perform batch deletion
    if indices_to_delete:
        deleted_dataset = deleted_dataset.drop(indices_to_delete)
        deleted_dataset = deleted_dataset.reset_index(drop=True)
    
    # Calculate actual deletion ratio
    actual_deletions = total_deletions
    actual_delete_ratio = actual_deletions / original_size
    
    print(f"\n  Deletion completed:")
    print(f"    Target deletions: {target_deletions}")
    print(f"    Actual deletions: {actual_deletions}")
    print(f"    Actual deletion ratio: {actual_delete_ratio*100:.2f}%")
    
    # Verify final user and item counts
    final_user_counts = deleted_dataset['user_id'].value_counts()
    final_item_counts = deleted_dataset['item_id'].value_counts()
    
    print(f"\n  Final verification:")
    print(f"    Final users: {len(final_user_counts)}")
    print(f"    Final items: {len(final_item_counts)}")
    print(f"    Final user min interactions: {final_user_counts.min()}")
    print(f"    Final item min interactions: {final_item_counts.min()}")
    
    # Check if all users and items satisfy thresholds
    users_below_threshold = final_user_counts[final_user_counts < user_threshold]
    items_below_threshold = final_item_counts[final_item_counts < item_threshold]
    
    if len(users_below_threshold) == 0:
        print(f"  All users have interaction count ≥ {user_threshold}")
    else:
        print(f"  {len(users_below_threshold)} users have interaction count < {user_threshold}")
        return None, 0, 0
    
    if len(items_below_threshold) == 0:
        print(f"  All items have interaction count ≥ {item_threshold}")
    else:
        print(f"  {len(items_below_threshold)} items have interaction count < {item_threshold}")
        return None, 0, 0
    
    return deleted_dataset, total_deletions, original_size

def truncate_or_pad(seq):
    """Truncate or pad sequence to fixed length (maintain original logic)"""
    cur_seq_len = len(seq)
    if cur_seq_len > max_seq_len:
        return seq[-max_seq_len:], max_seq_len
    else:
        PAD = 0
        return seq + [PAD] * (max_seq_len - cur_seq_len), cur_seq_len

def ensure_python_types(data):
    """Ensure all data is Python native types instead of numpy types"""
    if isinstance(data, np.integer):
        return int(data)
    elif isinstance(data, np.floating):
        return float(data)
    elif isinstance(data, np.ndarray):
        return data.tolist()
    elif isinstance(data, list):
        return [ensure_python_types(item) for item in data]
    elif isinstance(data, dict):
        return {key: ensure_python_types(value) for key, value in data.items()}
    else:
        return data

# ==================== Main Process ====================

print("="*60)
print("Starting Yelp Dataset Processing (10% deletion, ensuring user and item thresholds)")
print("="*60)

# 1. Load inter.csv file
print("\n1. Loading inter.csv file...")
inter_file = os.path.join(input_path, 'inter.csv')

if not os.path.exists(inter_file):
    print(f"Error: Cannot find file {inter_file}")
    exit(1)

# Read inter.csv file
dataset = pd.read_csv(inter_file)

print(f"Original data statistics:")
print(f"  Data shape: {dataset.shape}")
print(f"  Column names: {dataset.columns.tolist()}")
print(f"  Number of users: {dataset['user_id'].nunique()}")
print(f"  Number of items: {dataset['item_id'].nunique()}")
print(f"  Number of interactions: {len(dataset)}")

# 2. Filter dataset (based on interaction frequency)
print("\n2. Filtering dataset...")
filtered_dataset = dataset.copy()
while True:
    ori_len = len(filtered_dataset)
    
    # Filter users with less than user_threshold interactions
    filtered_dataset = filtered_dataset[filtered_dataset['user_id'].map(filtered_dataset['user_id'].value_counts()) >= user_threshold]
    # Filter items with less than item_threshold interactions
    filtered_dataset = filtered_dataset[filtered_dataset['item_id'].map(filtered_dataset['item_id'].value_counts()) >= item_threshold]
    
    if len(filtered_dataset) == ori_len:
        break

print(f"\nFiltered data statistics:")
print(f"  Number of users: {filtered_dataset['user_id'].nunique()}")
print(f"  Number of items: {filtered_dataset['item_id'].nunique()}")
print(f"  Number of interactions: {len(filtered_dataset)}")
print(f"  User interaction range: {filtered_dataset['user_id'].value_counts().min()} ~ {filtered_dataset['user_id'].value_counts().max()}")
print(f"  Item interaction range: {filtered_dataset['item_id'].value_counts().min()} ~ {filtered_dataset['item_id'].value_counts().max()}")

# 3. Remap IDs (consistent with original code)
print("\n3. Remapping IDs...")
all_user = filtered_dataset.user_id
all_item = filtered_dataset.item_id

user_id, user_token = pd.factorize(all_user)
item_id, item_token = pd.factorize(all_item)

num_users = len(user_token) + 1  # 0 id is for PAD
num_items = len(item_token) + 1  # 0 id is for PAD

user_mapping_dict = {_: idx + 1 for idx, _ in enumerate(user_token)}  # 0 id is for PAD
item_mapping_dict = {_: idx + 1 for idx, _ in enumerate(item_token)}  # 0 id is for PAD

print(f"User mapping: {user_token.shape}")
print(f"Item mapping: {item_token.shape}")

filtered_dataset['user_id'] = filtered_dataset['user_id'].apply(lambda x: user_mapping_dict[x])
filtered_dataset['item_id'] = filtered_dataset['item_id'].apply(lambda x: item_mapping_dict[x])

# Check if domain column exists, if not add one
if 'domain' not in filtered_dataset.columns:
    print("Note: No domain column in data, adding default domain=0")
    filtered_dataset['domain'] = 0

# Ensure domain is integer type
filtered_dataset['domain'] = filtered_dataset['domain'].astype(int)

# Save clean item list and counts
clean_users = sorted(filtered_dataset['user_id'].unique())
clean_items = sorted(filtered_dataset['item_id'].unique())
clean_user_counts = filtered_dataset['user_id'].value_counts().to_dict()
clean_item_counts = filtered_dataset['item_id'].value_counts().to_dict()
print(f"Clean user count: {len(clean_users)}")
print(f"Clean item count: {len(clean_items)}")
print(f"Clean user ID range: {clean_users[0]} ~ {clean_users[-1]}")
print(f"Clean item ID range: {clean_items[0]} ~ {clean_items[-1]}")
print(f"Clean minimum user interactions: {min(clean_user_counts.values())}")
print(f"Clean minimum item interactions: {min(clean_item_counts.values())}")

# 4. Create output directory
os.makedirs(output_path, exist_ok=True)

# 5. Process clean data
print(f"\n{'='*40}")
print(f"Processing clean data...")
print(f"{'='*40}")

clean_dataset = filtered_dataset.copy()
clean_type = 'clean'

# Create output directory
clean_dir = os.path.join(output_path, dataset_name, clean_type)
os.makedirs(clean_dir, exist_ok=True)

# Save inter.csv
csv_path = os.path.join(clean_dir, 'inter.csv')
clean_dataset.to_csv(csv_path, sep=',', index=None)
print(f"  Saving {clean_type} interaction data to: {csv_path}")

# Generate sequence data following original logic
print(f"  Generating sequence data...")

# Sort by user and time (original code logic)
clean_dataset_sorted = clean_dataset.sort_values(by=['user_id', 'timestamp'])

# ==================== Generate seq2pat_data.pth ====================
def to_list(x):
    return list(x)[:-2]  # Remove last 2 interactions, maintain original logic

user_group_for_seq2pat = clean_dataset_sorted.groupby('user_id')['item_id'].apply(to_list)

# Filter out empty lists
seq2pat_data = [seq for seq in user_group_for_seq2pat.tolist() if len(seq) > 0]

# Ensure data types are Python native types
seq2pat_data = ensure_python_types(seq2pat_data)

# Save seq2pat_data.pth
seq2pat_path = os.path.join(clean_dir, 'seq2pat_data.pth')
torch.save(seq2pat_data, seq2pat_path)
print(f"  Generated {clean_type}/seq2pat_data.pth, valid sequences: {len(seq2pat_data)}")
# ==================== seq2pat_data generation ends ====================

# Continue generating train/val/test sequences
user_group = clean_dataset_sorted.groupby('user_id')['item_id'].apply(list)

# Initialize lists
train, val, test = [], [], []

PAD = 0

# Process each user
for user_id, user_seq in list(zip(user_group.index, user_group.tolist())):
    # Ensure user_id is Python int type
    user_id = int(user_id)
    
    # Truncate to maximum sequence length
    user_seq = user_seq[-max_seq_len:]
    
    # Skip users with sequences too short
    if len(user_seq) < 3:
        continue
        
    # Ensure item_id in user_seq is Python int type
    user_seq = [int(item) for item in user_seq]
        
    # ------ Test sample ------------
    history, seq_len = truncate_or_pad(user_seq[:-1])
    target_data = user_seq[-1]
    label = 1
    
    # Domain handling: following original code logic, domain_id is [0] * max_seq_len
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    test.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Validation sample -------------
    history, seq_len = truncate_or_pad(user_seq[:-2])
    target_data = user_seq[-2]
    label = 1
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    val.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Training sample -----------
    # Maintain original logic
    history, seq_len = truncate_or_pad(user_seq[:-3])
    target_data, _ = truncate_or_pad(user_seq[-seq_len-2:-2])
    label = [1] * seq_len + [PAD] * (max_seq_len - seq_len)
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    train.append([user_id, history, target_data, seq_len, label, domain_id])

# Ensure all data are Python native types
train = ensure_python_types(train)
val = ensure_python_types(val)
test = ensure_python_types(test)

# Save sequence data
torch.save(train, os.path.join(clean_dir, 'train.pth'))
torch.save(val, os.path.join(clean_dir, 'val.pth'))
torch.save(test, os.path.join(clean_dir, 'test.pth'))

print(f"  Saved sequence data: training={len(train)}, validation={len(val)}, test={len(test)}")

# 6. Generate 10% deletion data
print(f"\n{'='*40}")
print(f"Processing 10% deletion data (ensuring user and item thresholds)...")
print(f"{'='*40}")

# Generate deletion data
result = delete_interactions_guarantee_thresholds(
    filtered_dataset.copy(), 
    delete_ratio=DELETE_RATIO
)

if result[0] is None:
    print("Deletion failed, cannot satisfy all conditions!")
    # Create an empty deletion dataset but mark as failed
    deleted_type = f'delete_{int(DELETE_RATIO*100)}_FAILED'
    deleted_dir = os.path.join(output_path, dataset_name, deleted_type)
    os.makedirs(deleted_dir, exist_ok=True)
    
    # Save error message
    with open(os.path.join(deleted_dir, 'ERROR.txt'), 'w') as f:
        f.write(f"Deletion operation failed: Cannot delete {DELETE_RATIO*100}% of interactions while ensuring all users and items have interaction count ≥ {item_threshold}.")
    
    print(f"  Created error marker directory: {deleted_dir}")
else:
    # Unpack result
    deleted_dataset, total_deletions, original_size = result
    
    deleted_type = f'delete_{int(DELETE_RATIO*100)}'
    
    # Create output directory
    deleted_dir = os.path.join(output_path, dataset_name, deleted_type)
    os.makedirs(deleted_dir, exist_ok=True)
    
    # Save inter.csv
    deleted_csv_path = os.path.join(deleted_dir, 'inter.csv')
    deleted_dataset.to_csv(deleted_csv_path, sep=',', index=None)
    print(f"  Saving {deleted_type} interaction data to: {deleted_csv_path}")
    
    # Generate sequence data following original logic
    print(f"  Generating sequence data...")
    
    # Sort by user and time (original code logic)
    deleted_dataset_sorted = deleted_dataset.sort_values(by=['user_id', 'timestamp'])
    
    # ==================== Generate seq2pat_data.pth ====================
    user_group_for_seq2pat_deleted = deleted_dataset_sorted.groupby('user_id')['item_id'].apply(to_list)
    
    # Filter out empty lists
    seq2pat_data_deleted = [seq for seq in user_group_for_seq2pat_deleted.tolist() if len(seq) > 0]
    
    # Ensure data types are Python native types
    seq2pat_data_deleted = ensure_python_types(seq2pat_data_deleted)
    
    # Save seq2pat_data.pth
    seq2pat_path_deleted = os.path.join(deleted_dir, 'seq2pat_data.pth')
    torch.save(seq2pat_data_deleted, seq2pat_path_deleted)
    print(f"  Generated {deleted_type}/seq2pat_data.pth, valid sequences: {len(seq2pat_data_deleted)}")
    # ==================== seq2pat_data generation ends ====================
    
    # Continue generating train/val/test sequences
    user_group_deleted = deleted_dataset_sorted.groupby('user_id')['item_id'].apply(list)
    
    # Initialize lists
    train_deleted, val_deleted, test_deleted = [], [], []
    
    # Process each user
    for user_id, user_seq in list(zip(user_group_deleted.index, user_group_deleted.tolist())):
        # Ensure user_id is Python int type
        user_id = int(user_id)
        
        # Truncate to maximum sequence length
        user_seq = user_seq[-max_seq_len:]
        
        # Skip users with sequences too short
        if len(user_seq) < 3:
            continue
            
        # Ensure item_id in user_seq is Python int type
        user_seq = [int(item) for item in user_seq]
            
        # ------ Test sample ------------
        history, seq_len = truncate_or_pad(user_seq[:-1])
        target_data = user_seq[-1]
        label = 1
        
        # Domain handling: following original code logic, domain_id is [0] * max_seq_len
        domain_id = [0] * max_seq_len
        
        # Ensure data types
        history = ensure_python_types(history)
        target_data = ensure_python_types(target_data)
        seq_len = ensure_python_types(seq_len)
        label = ensure_python_types(label)
        domain_id = ensure_python_types(domain_id)
        
        test_deleted.append([user_id, history, target_data, seq_len, label, domain_id, history])
        
        # ------ Validation sample -------------
        history, seq_len = truncate_or_pad(user_seq[:-2])
        target_data = user_seq[-2]
        label = 1
        domain_id = [0] * max_seq_len
        
        # Ensure data types
        history = ensure_python_types(history)
        target_data = ensure_python_types(target_data)
        seq_len = ensure_python_types(seq_len)
        label = ensure_python_types(label)
        domain_id = ensure_python_types(domain_id)
        
        val_deleted.append([user_id, history, target_data, seq_len, label, domain_id, history])
        
        # ------ Training sample -----------
        history, seq_len = truncate_or_pad(user_seq[:-3])
        target_data, _ = truncate_or_pad(user_seq[-seq_len-2:-2])
        label = [1] * seq_len + [PAD] * (max_seq_len - seq_len)
        domain_id = [0] * max_seq_len
        
        # Ensure data types
        history = ensure_python_types(history)
        target_data = ensure_python_types(target_data)
        seq_len = ensure_python_types(seq_len)
        label = ensure_python_types(label)
        domain_id = ensure_python_types(domain_id)
        
        train_deleted.append([user_id, history, target_data, seq_len, label, domain_id])
    
    # Ensure all data are Python native types
    train_deleted = ensure_python_types(train_deleted)
    val_deleted = ensure_python_types(val_deleted)
    test_deleted = ensure_python_types(test_deleted)
    
    # Save sequence data
    torch.save(train_deleted, os.path.join(deleted_dir, 'train.pth'))
    torch.save(val_deleted, os.path.join(deleted_dir, 'val.pth'))
    torch.save(test_deleted, os.path.join(deleted_dir, 'test.pth'))
    
    print(f"  Saved sequence data: training={len(train_deleted)}, validation={len(val_deleted)}, test={len(test_deleted)}")

print("\n" + "="*60)
print("Data processing completed!")
print("="*60)

# 7. Final verification
print(f"\n{'='*40}")
print("Final verification results:")
print(f"{'='*40}")

if result[0] is not None:
    # Verify data integrity
    df_clean = pd.read_csv(csv_path)
    df_deleted = pd.read_csv(deleted_csv_path)
    
    print(f"Clean data: {len(df_clean)} rows, {df_clean['user_id'].nunique()} users, {df_clean['item_id'].nunique()} items")
    print(f"Delete data: {len(df_deleted)} rows, {df_deleted['user_id'].nunique()} users, {df_deleted['item_id'].nunique()} items")
    
    # Verify user thresholds
    clean_user_min = df_clean['user_id'].value_counts().min()
    clean_item_min = df_clean['item_id'].value_counts().min()
    deleted_user_min = df_deleted['user_id'].value_counts().min()
    deleted_item_min = df_deleted['item_id'].value_counts().min()
    
    print(f"\n  User min interactions: Clean={clean_user_min}, Delete={deleted_user_min}")
    print(f"  Item min interactions: Clean={clean_item_min}, Delete={deleted_item_min}")
    
    if deleted_user_min >= user_threshold:
        print(f"  All users have interaction count ≥ {user_threshold}")
    else:
        print(f"  Some users have interaction count < {user_threshold}")
    
    if deleted_item_min >= item_threshold:
        print(f"  All items have interaction count ≥ {item_threshold}")
    else:
        print(f"  Some items have interaction count < {item_threshold}")
    
    # Calculate actual deletion ratio
    actual_delete_ratio = (len(df_clean) - len(df_deleted)) / len(df_clean)
    print(f"\nDeletion ratio statistics:")
    print(f"  Target deletion ratio: {DELETE_RATIO*100:.1f}%")
    print(f"  Actual deletion ratio: {actual_delete_ratio*100:.2f}%")
    print(f"  Absolute error: {abs(actual_delete_ratio - DELETE_RATIO)*100:.3f}%")

print(f"\nOutput directory structure:")
print(f"{output_path}/")
print(f"└── {dataset_name}/")
print(f"    ├── clean/")
print(f"    │   ├── inter.csv")
print(f"    │   ├── seq2pat_data.pth")
print(f"    │   ├── train.pth")
print(f"    │   ├── val.pth")
print(f"    │   └── test.pth")

if result[0] is not None:
    print(f"    └── delete_10/")
    print(f"        ├── inter.csv")
    print(f"        ├── seq2pat_data.pth")
    print(f"        ├── train.pth")
    print(f"        ├── val.pth")
    print(f"        └── test.pth")

print(f"\nKey guarantees:")
print(f"1. All users have interaction count ≥ {user_threshold}")
print(f"2. All items have interaction count ≥ {item_threshold}")
print(f"3. Precisely controlled deletion ratio ≈ 10%")
print(f"4. Delete entire interaction records")
print(f"5. Domain handled according to original code logic (fixed to 0)")
print(f"6. All data are Python native types, avoiding numpy type issues")

print("\nYelp dataset deletion experiment ready!")